In [ ]:
import os
import re
import urllib.request
import math
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader

In [ ]:
batch_size = 64
block_size = 24  # Max sequence length (tokens)
max_iters = 3000  # Total training iterations
eval_interval = 300
learning_rate = 5e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 100

In [ ]:
n_embd = 128           # Embedding dimension
n_head = 4             # no. of attention heads
n_layer = 3            # no. of encoder-decoder layers
dropout = 0.1
max_vocab_size = 50000

print(f"Using device: {device}")

Using device: cuda


In [ ]:
#dataset
DATA_URL = "https://raw.githubusercontent.com/multi30k/dataset/master/data/task1/tok/"
ENG_FILE = "train.lc.norm.tok.en"
FRA_FILE = "train.lc.norm.tok.fr"

def download_data():
    if not os.path.exists(ENG_FILE):
        print("Downloading English training data...")
        urllib.request.urlretrieve(DATA_URL + ENG_FILE, ENG_FILE)
    if not os.path.exists(FRA_FILE):
        print("Downloading French training data...")
        urllib.request.urlretrieve(DATA_URL + FRA_FILE, FRA_FILE)

download_data()

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, eng_path, fra_path, max_vocab, max_len):
        self.max_len = max_len

        with open(eng_path, 'r', encoding='utf-8') as f:
            eng_lines = [self.clean(l) for l in f.read().strip().split('\n')]
        with open(fra_path, 'r', encoding='utf-8') as f:
            fra_lines = [self.clean(l) for l in f.read().strip().split('\n')]

        # Filter sequences that fit inside our block_size constraint
        self.pairs = []
        for eng, fra in zip(eng_lines, fra_lines):
            if len(eng.split()) <= max_len - 2 and len(fra.split()) <= max_len - 2:
                self.pairs.append((eng, fra))

        # Build independent vocabularies for Source and Target
        self.eng_vocab, self.inv_eng_vocab = self.build_vocab([p[0] for p in self.pairs], max_vocab)
        self.fra_vocab, self.inv_fra_vocab = self.build_vocab([p[1] for p in self.pairs], max_vocab)

    def clean(self, text):
        text = text.lower().strip()
        text = re.sub(r"([?.!,¿])", r" \1 ", text)
        return re.sub(r'[" "]+', " ", text).strip()

    def build_vocab(self, sentences, max_size):
        vocab = {'<pad>': 0, '<unk>': 1, '<sos>': 2, '<eos>': 3}
        counts = {}
        for s in sentences:
            for word in s.split():
                counts[word] = counts.get(word, 0) + 1
        sorted_words = sorted(counts.items(), key=lambda x: x[1], reverse=True)
        for w, _ in sorted_words[:max_size - 4]:
            vocab[w] = len(vocab)
        inv_vocab = {v: k for k, v in vocab.items()}
        return vocab, inv_vocab

    def encode(self, sentence, vocab):
        tokens = [vocab['<sos>']] + [vocab.get(w, vocab['<unk>']) for w in sentence.split()] + [vocab['<eos>']]
        if len(tokens) < self.max_len:
            tokens += [vocab['<pad>']] * (self.max_len - len(tokens))
        return torch.tensor(tokens[:self.max_len], dtype=torch.long)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        eng_s, fra_s = self.pairs[idx]
        return self.encode(eng_s, self.eng_vocab), self.encode(fra_s, self.fra_vocab)

dataset = TranslationDataset(ENG_FILE, FRA_FILE, max_vocab_size, block_size)
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, drop_last=True)

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_head, n_embd, is_causal=False):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.is_causal = is_causal

        self.q_linear = nn.Linear(n_embd, n_embd, bias=False)
        self.k_linear = nn.Linear(n_embd, n_embd, bias=False)
        self.v_linear = nn.Linear(n_embd, n_embd, bias=False)
        self.out_linear = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

        if self.is_causal:
            # Register a lower-triangular causal matrix for decoder self-attention
            self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x_q, x_k, x_v):
        B, T_q, C = x_q.shape
        _, T_k, _ = x_k.shape

        # Linear project and project to heads: (B, T, n_head, head_dim) -> transpose to (B, n_head, T, head_dim)
        q = self.q_linear(x_q).view(B, T_q, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_linear(x_k).view(B, T_k, self.n_head, self.head_dim).transpose(1, 2)
        v = self.v_linear(x_v).view(B, T_k, self.n_head, self.head_dim).transpose(1, 2)

        # Calculate scaled attention: (B, nhead, Tq, headdim) x (B, nhead, headdim, Tk)
        wei = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)

        if self.is_causal:
            # Prevent look-ahead in Decoder self-attention
            wei = wei.masked_fill(self.tril[:T_q, :T_k] == 0, float('-inf'))

        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        out = wei @ v # (B, n_head, T_q, head_dim)
        out = out.transpose(1, 2).contiguous().view(B, T_q, C) #recombinatoin
        return self.out_linear(out)

In [ ]:
#encoding decoding blocks:
class EncoderBlock(nn.Module):
    """ Encapsulates Multi-Head Self-Attention and standard MLP with Pre-LN residuals """
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.sa = MultiHeadAttention(n_head, n_embd, is_causal=False)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffwd = FeedForward(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x), self.ln1(x), self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [ ]:
class DecoderBlock(nn.Module):
    """ Decodes by attending to both generated targets (Causal) and encoder memory states """
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.sa = MultiHeadAttention(n_head, n_embd, is_causal=True) # Self Attention Masked
        self.ln2 = nn.LayerNorm(n_embd)
        self.ca = MultiHeadAttention(n_head, n_embd, is_causal=False) # Cross Attention
        self.ln3 = nn.LayerNorm(n_embd)
        self.ffwd = FeedForward(n_embd)

    def forward(self, x, enc_mem):
        x = x + self.sa(self.ln1(x), self.ln1(x), self.ln1(x))
        x = x + self.ca(self.ln2(x), enc_mem, enc_mem) # Query from decoder, Keys/Values from encoder
        x = x + self.ffwd(self.ln3(x))
        return x

In [ ]:
class TransformerSeq2Seq(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size):
        super().__init__()
        self.src_tok_emb = nn.Embedding(src_vocab_size, n_embd)
        self.tgt_tok_emb = nn.Embedding(tgt_vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)

        self.encoder_stack = nn.ModuleList([EncoderBlock(n_embd, n_head) for _ in range(n_layer)])
        self.decoder_stack = nn.ModuleList([DecoderBlock(n_embd, n_head) for _ in range(n_layer)])

        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, tgt_vocab_size)

    def forward(self, src, tgt):
        B, T_src = src.shape
        _, T_tgt = tgt.shape

        # 1. Process Source inputs into structural latent space (Encoder)
        pos_src = torch.arange(0, T_src, dtype=torch.long, device=device).unsqueeze(0)
        enc_out = self.src_tok_emb(src) + self.pos_emb(pos_src)
        for layer in self.encoder_stack:
            enc_out = layer(enc_out)

        # 2. Process Target shifts through causal & cross features (Decoder)
        pos_tgt = torch.arange(0, T_tgt, dtype=torch.long, device=device).unsqueeze(0)
        dec_out = self.tgt_tok_emb(tgt) + self.pos_emb(pos_tgt)
        for layer in self.decoder_stack:
            dec_out = layer(dec_out, enc_out)

        # 3. Predict mapping classes across structural targets
        dec_out = self.ln_f(dec_out)
        logits = self.lm_head(dec_out)
        return logits


In [ ]:
model = TransformerSeq2Seq(len(dataset.eng_vocab), len(dataset.fra_vocab)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [ ]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split, loader in [('train', train_loader), ('val', val_loader)]:
        losses = torch.zeros(eval_iters)
        for k, (src, tgt) in enumerate(loader):
            if k >= eval_iters: break
            src, tgt = src.to(device), tgt.to(device)

            # For decoding targets: input sequence is 0 to T-1
            tgt_input = tgt[:, :-1]
            tgt_targets = tgt[:, 1:]

            logits = model(src, tgt_input)
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.reshape(B*T, C), tgt_targets.reshape(B*T), ignore_index=0)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:
train_iter = iter(train_loader)
for step in range(max_iters):
    try:
        src, tgt = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        src, tgt = next(train_iter)

    src, tgt = src.to(device), tgt.to(device)
    tgt_input = tgt[:, :-1]
    tgt_targets = tgt[:, 1:]

    logits = model(src, tgt_input)
    B, T, C = logits.shape
    loss = F.cross_entropy(logits.reshape(B*T, C), tgt_targets.reshape(B*T), ignore_index=0)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % eval_interval == 0 or step == max_iters - 1:
        losses = estimate_loss()
        print(f"step {step}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

step 0: train loss 9.0220, val loss 3.7927
step 300: train loss 3.3557, val loss 1.4439
step 600: train loss 2.5653, val loss 1.1390
step 900: train loss 2.0831, val loss 0.9668
step 1200: train loss 1.7407, val loss 0.8612
step 1500: train loss 1.5013, val loss 0.7900
step 1800: train loss 1.2945, val loss 0.7411
step 2100: train loss 1.1240, val loss 0.7057
step 2400: train loss 0.9884, val loss 0.6813
step 2700: train loss 0.8501, val loss 0.6617
step 2999: train loss 0.7413, val loss 0.6540


In [ ]:
def translate(sentence):
    model.eval()
    cleaned = dataset.clean(sentence)
    src_tokens = dataset.encode(cleaned, dataset.eng_vocab).unsqueeze(0).to(device)

    # Target initializes with the <sos> token
    tgt_tokens = torch.tensor([[dataset.fra_vocab['<sos>']]], dtype=torch.long, device=device)

    for _ in range(block_size):
        with torch.no_grad():
            logits = model(src_tokens, tgt_tokens)
        next_token_logits = logits[:, -1, :]
        next_token = torch.argmax(next_token_logits, dim=-1).unsqueeze(0)

        tgt_tokens = torch.cat([tgt_tokens, next_token], dim=1)
        if next_token.item() == dataset.fra_vocab['<eos>'] or tgt_tokens.shape[1] >= block_size:
            break


    # Decode back into readable characters
    translated_words = [dataset.inv_fra_vocab[idx] for idx in tgt_tokens[0].cpu().numpy()]
    filtered = [w for w in translated_words if w not in ['<sos>', '<eos>', '<pad>']]
    return " ".join(filtered)

In [ ]:
sentences = ["dogs are animals .", "she loves black cats .", "a person is walking ."]
for s in sentences:
    print(f"Input:  {s}")
    print(f"Output: {translate(s)}\n")

Input:  dogs are animals .
Output: des chiens font des animaux .

Input:  she loves black cats .
Output: elle monte des chats .

Input:  a person is walking .
Output: une personne marche .

